In [ ]:
# Install all required libraries — every notebook is self-contained
!pip install fastapi uvicorn scikit-fem nest-asyncio requests pyngrok meshio -q

import nest_asyncio
nest_asyncio.apply()

print("✅ Libraries installed and nest-asyncio applied")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.2/166.2 kB 6.6 MB/s eta 0:00:00
✅ Libraries installed and nest-asyncio applied


In [ ]:
import os
import json
import time
import logging
import threading
import requests
import uvicorn
from datetime import datetime
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define paths
DRIVE_PATH = '/content/drive/MyDrive/MCL_Orchestrator/'

PATHS = {
    'notebooks' : f'{DRIVE_PATH}notebooks/',
    'services'  : f'{DRIVE_PATH}services/',
    'results'   : f'{DRIVE_PATH}results/',
    'logs'      : f'{DRIVE_PATH}logs/',
    'docs'      : f'{DRIVE_PATH}docs/'
}

# Confirm folders
print("\nConfirming Drive folder structure:")
print("=" * 45)
all_exist = True
for name, path in PATHS.items():
    exists = os.path.exists(path)
    status = "✅ Found" if exists else "❌ Missing"
    print(f"  {name:<12} : {status}")
    if not exists:
        all_exist = False

print("=" * 45)
if all_exist:
    print("✅ All folders confirmed")
else:
    print("❌ Some folders missing — check Drive before continuing")

Mounted at /content/drive

Confirming Drive folder structure:
  notebooks    : ✅ Found
  services     : ✅ Found
  results      : ✅ Found
  logs         : ✅ Found
  docs         : ✅ Found
✅ All folders confirmed


In [ ]:
# ── Logger ────────────────────────────────────────────────────────
log_file_path = f'{DRIVE_PATH}logs/pipeline.log'

# Create log file if it doesn't exist
if not os.path.exists(log_file_path):
    with open(log_file_path, 'w') as f:
        f.write(f"MCL Orchestrator — Pipeline Log\n")
        f.write(f"Created : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 60 + "\n")

# Named logger with handler guard — prevents triple printing on re-run
logger = logging.getLogger('MCL_Orchestrator')
logger.setLevel(logging.INFO)

if not logger.handlers:
    # File handler
    file_handler = logging.FileHandler(log_file_path)
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

def log_event(stage, status, detail=''):
    message = f"{stage:<30} | {status:<10} | {detail}"
    if status == 'ERROR':
        logger.error(message)
    elif status == 'WARNING':
        logger.warning(message)
    else:
        logger.info(message)

# ── Pipeline State ────────────────────────────────────────────────
state_file_path = f'{DRIVE_PATH}logs/pipeline_state.json'

pipeline_state = {
    'job_id'      : None,
    'started_at'  : None,
    'stages'      : {
        'A_parameter_generation': 'pending',
        'B_fem_simulation'      : 'pending',
        'C_postprocessing'      : 'pending'
    },
    'errors'      : [],
    'completed_at': None
}

def update_state(stage, status, error=None):
    pipeline_state['stages'][stage] = status
    if error:
        pipeline_state['errors'].append({
            'stage'    : stage,
            'error'    : str(error),
            'timestamp': datetime.now().isoformat()
        })
        log_event(stage, 'ERROR', str(error))
    else:
        log_event(stage, status, f"Stage updated to {status}")
    with open(state_file_path, 'w') as f:
        json.dump(pipeline_state, f, indent=4)

# ── Retry Handler ─────────────────────────────────────────────────
def post_with_retry(url, data, max_retries=3, backoff=2):
    log_event('retry_handler', 'INFO',
              f"POST to {url} | max_retries={max_retries}")
    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=data, timeout=30)
            response.raise_for_status()
            log_event('retry_handler', 'SUCCESS',
                      f"POST succeeded on attempt {attempt + 1}")
            return response.json()
        except requests.exceptions.ConnectionError:
            if attempt < max_retries - 1:
                wait = backoff ** attempt
                print(f"  ⚠️  Service unreachable — retrying in {wait}s "
                      f"(attempt {attempt + 1}/{max_retries})")
                log_event('retry_handler', 'WARNING',
                          f"Connection failed — retrying in {wait}s")
                time.sleep(wait)
            else:
                log_event('retry_handler', 'ERROR',
                          f"Unreachable after {max_retries} attempts")
                raise RuntimeError(
                    f"❌ Service at {url} unreachable after {max_retries} attempts"
                )
        except requests.exceptions.Timeout:
            log_event('retry_handler', 'ERROR', f"Timeout at {url}")
            raise RuntimeError(f"❌ Service at {url} timed out")
        except requests.exceptions.HTTPError as e:
            log_event('retry_handler', 'ERROR', f"HTTP error: {e}")
            raise RuntimeError(f"❌ HTTP error: {e}")

# ── Confirmation ──────────────────────────────────────────────────
print("✅ Shared utilities reinitialised:")
print("   Logger        : ✅ handler guard active — no duplicate printing")
print("   State tracker : ✅ ready")
print("   Retry handler : ✅ ready")
log_event('01_service_A', 'INFO', 'Shared utilities initialised in Notebook 2')

✅ Shared utilities reinitialised:
   Logger        : ✅ handler guard active — no duplicate printing
   State tracker : ✅ ready
   Retry handler : ✅ ready


2026-06-01 17:08:11 | INFO | 01_service_A                   | INFO       | Shared utilities initialised in Notebook 2
INFO:MCL_Orchestrator:01_service_A                   | INFO       | Shared utilities initialised in Notebook 2


In [ ]:
class ParameterRequest(BaseModel):
    E_fiber_GPa           : float = 380.0  # SiC fiber modulus (GPa)
    E_matrix_GPa          : float = 90.0   # SiC matrix modulus (GPa)
    fiber_volume_fraction : float = 0.45   # Fiber volume fraction (0-1)
    nu                    : float = 0.20   # Poisson's ratio
    applied_stress_MPa    : float = 300.0  # Applied tensile stress (MPa)
    length_mm             : float = 100.0  # Coupon length (mm)
    width_mm              : float = 20.0   # Coupon width (mm)

# Print schema with defaults
print("Input Parameter Schema — ParameterRequest")
print("=" * 50)
defaults = ParameterRequest()
fields = {
    'E_fiber_GPa'          : ('SiC fiber Young\'s modulus',  'GPa',  defaults.E_fiber_GPa),
    'E_matrix_GPa'         : ('SiC matrix Young\'s modulus', 'GPa',  defaults.E_matrix_GPa),
    'fiber_volume_fraction': ('Fiber volume fraction',        '-',    defaults.fiber_volume_fraction),
    'nu'                   : ('Poisson\'s ratio',             '-',    defaults.nu),
    'applied_stress_MPa'   : ('Applied tensile stress',       'MPa',  defaults.applied_stress_MPa),
    'length_mm'            : ('Coupon length',                'mm',   defaults.length_mm),
    'width_mm'             : ('Coupon width',                 'mm',   defaults.width_mm)
}

for field, (description, unit, default) in fields.items():
    print(f"  {field:<25} : {default:<8} {unit:<5} — {description}")

print("=" * 50)
print("✅ ParameterRequest schema defined with SiC/SiC defaults")

Input Parameter Schema — ParameterRequest
  E_fiber_GPa               : 380.0    GPa   — SiC fiber Young's modulus
  E_matrix_GPa              : 90.0     GPa   — SiC matrix Young's modulus
  fiber_volume_fraction     : 0.45     -     — Fiber volume fraction
  nu                        : 0.2      -     — Poisson's ratio
  applied_stress_MPa        : 300.0    MPa   — Applied tensile stress
  length_mm                 : 100.0    mm    — Coupon length
  width_mm                  : 20.0     mm    — Coupon width
✅ ParameterRequest schema defined with SiC/SiC defaults


In [ ]:
def compute_effective_modulus(E_fiber_GPa, E_matrix_GPa, Vf):
    """
    Computes the effective Young's modulus of a composite
    using the Voigt upper bound (rule of mixtures).

    Voigt assumes load is parallel to fibers — gives the
    upper bound stiffness of the composite system.

    Args:
        E_fiber_GPa : fiber Young's modulus in GPa
        E_matrix_GPa: matrix Young's modulus in GPa
        Vf          : fiber volume fraction (0-1)

    Returns:
        E_eff_GPa   : effective composite modulus in GPa
    """
    E_eff = Vf * E_fiber_GPa + (1 - Vf) * E_matrix_GPa
    return round(E_eff, 3)

# Test with standard SiC/SiC values
E_fiber = 380.0
E_matrix = 90.0
Vf = 0.45

E_eff = compute_effective_modulus(E_fiber, E_matrix, Vf)

# Expected SiC/SiC effective modulus range: 150-280 GPa
in_range = 150 <= E_eff <= 280

print("Voigt Rule of Mixtures — Test")
print("=" * 45)
print(f"  E_fiber          : {E_fiber} GPa")
print(f"  E_matrix         : {E_matrix} GPa")
print(f"  Fiber vol. frac. : {Vf}")
print(f"  E_eff (Voigt)    : {E_eff} GPa")
print(f"  Expected range   : 150 — 280 GPa")
print(f"  In range         : {'✅ Yes' if in_range else '❌ No — check inputs'}")
print("=" * 45)

# Sensitivity check — show E_eff across Vf range
print("\n  E_eff vs Fiber Volume Fraction:")
print("  " + "-" * 35)
for vf_test in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    e = compute_effective_modulus(E_fiber, E_matrix, vf_test)
    bar = "█" * int(e / 10)
    print(f"  Vf={vf_test:.2f} → E_eff={e:>7.1f} GPa  {bar}")
print("  " + "-" * 35)
print("\n✅ Voigt function confirmed")

Voigt Rule of Mixtures — Test
  E_fiber          : 380.0 GPa
  E_matrix         : 90.0 GPa
  Fiber vol. frac. : 0.45
  E_eff (Voigt)    : 220.5 GPa
  Expected range   : 150 — 280 GPa
  In range         : ✅ Yes

  E_eff vs Fiber Volume Fraction:
  -----------------------------------
  Vf=0.30 → E_eff=  177.0 GPa  █████████████████
  Vf=0.35 → E_eff=  191.5 GPa  ███████████████████
  Vf=0.40 → E_eff=  206.0 GPa  ████████████████████
  Vf=0.45 → E_eff=  220.5 GPa  ██████████████████████
  Vf=0.50 → E_eff=  235.0 GPa  ███████████████████████
  Vf=0.55 → E_eff=  249.5 GPa  ████████████████████████
  Vf=0.60 → E_eff=  264.0 GPa  ██████████████████████████
  Vf=0.65 → E_eff=  278.5 GPa  ███████████████████████████
  Vf=0.70 → E_eff=  293.0 GPa  █████████████████████████████
  -----------------------------------

✅ Voigt function confirmed


In [ ]:
def validate_parameters(p: ParameterRequest):
    """
    Validates all input parameters against physically realistic
    ranges for SiC/SiC composite materials.

    Raises HTTPException 422 if any parameter is out of range.
    All violations are collected and returned together —
    not one at a time — so the caller sees all issues at once.
    """
    errors = []

    # E_fiber range — SiC fibers typically 200-450 GPa
    if not (100 <= p.E_fiber_GPa <= 500):
        errors.append(
            f"E_fiber_GPa={p.E_fiber_GPa} out of range — "
            f"expected 100-500 GPa for ceramic fibers"
        )

    # E_matrix range — SiC matrix typically 50-150 GPa
    if not (50 <= p.E_matrix_GPa <= 200):
        errors.append(
            f"E_matrix_GPa={p.E_matrix_GPa} out of range — "
            f"expected 50-200 GPa for ceramic matrix"
        )

    # Fiber volume fraction — physically bounded 0.3-0.7
    if not (0.3 <= p.fiber_volume_fraction <= 0.7):
        errors.append(
            f"fiber_volume_fraction={p.fiber_volume_fraction} out of range — "
            f"expected 0.3-0.7 for SiC/SiC composites"
        )

    # Poisson's ratio — ceramics typically 0.1-0.35
    if not (0.1 <= p.nu <= 0.45):
        errors.append(
            f"nu={p.nu} out of range — "
            f"expected 0.1-0.45 for ceramic composites"
        )

    # Applied stress — must be positive for tensile test
    if p.applied_stress_MPa <= 0:
        errors.append(
            f"applied_stress_MPa={p.applied_stress_MPa} invalid — "
            f"must be positive for a tensile test"
        )

    # Realistic stress range for SiC/SiC
    if p.applied_stress_MPa > 1500:
        errors.append(
            f"applied_stress_MPa={p.applied_stress_MPa} exceeds "
            f"realistic SiC/SiC UTS range — max 1500 MPa"
        )

    # Coupon dimensions — must be positive
    if p.length_mm <= 0:
        errors.append(
            f"length_mm={p.length_mm} invalid — must be positive"
        )
    if p.width_mm <= 0:
        errors.append(
            f"width_mm={p.width_mm} invalid — must be positive"
        )

    # Aspect ratio check — length must be greater than width
    if p.length_mm <= p.width_mm:
        errors.append(
            f"Coupon aspect ratio invalid — "
            f"length ({p.length_mm}mm) must exceed width ({p.width_mm}mm)"
        )

    if errors:
        log_event('Service A — validation', 'ERROR',
                  f"{len(errors)} validation error(s) detected")
        raise HTTPException(status_code=422, detail=errors)

    log_event('Service A — validation', 'SUCCESS',
              'All parameters within physical range')

# ── Test 1 — Valid SiC/SiC parameters ────────────────────────────
print("Test 1 — Valid SiC/SiC parameters:")
print("-" * 45)
try:
    valid_params = ParameterRequest()
    validate_parameters(valid_params)
    print("✅ Valid parameters passed — no errors raised")
except HTTPException as e:
    print(f"❌ Unexpected failure: {e.detail}")

# ── Test 2 — Invalid parameters ───────────────────────────────────
print("\nTest 2 — Invalid parameters:")
print("-" * 45)
try:
    invalid_params = ParameterRequest(
        E_fiber_GPa           = 999.0,   # too high
        E_matrix_GPa          = 90.0,
        fiber_volume_fraction = 0.95,    # out of range
        nu                    = 0.20,
        applied_stress_MPa    = -100.0,  # negative
        length_mm             = 10.0,    # shorter than width
        width_mm              = 20.0
    )
    validate_parameters(invalid_params)
except HTTPException as e:
    print(f"✅ Validation correctly rejected invalid parameters:")
    for i, error in enumerate(e.detail, 1):
        print(f"   {i}. {error}")

print("\n✅ Input validation confirmed")

2026-06-01 17:51:21 | INFO | Service A — validation         | SUCCESS    | All parameters within physical range
INFO:MCL_Orchestrator:Service A — validation         | SUCCESS    | All parameters within physical range
2026-06-01 17:51:21 | ERROR | Service A — validation         | ERROR      | 4 validation error(s) detected
ERROR:MCL_Orchestrator:Service A — validation         | ERROR      | 4 validation error(s) detected


Test 1 — Valid SiC/SiC parameters:
---------------------------------------------
✅ Valid parameters passed — no errors raised

Test 2 — Invalid parameters:
---------------------------------------------
✅ Validation correctly rejected invalid parameters:
   1. E_fiber_GPa=999.0 out of range — expected 100-500 GPa for ceramic fibers
   2. fiber_volume_fraction=0.95 out of range — expected 0.3-0.7 for SiC/SiC composites
   3. applied_stress_MPa=-100.0 invalid — must be positive for a tensile test
   4. Coupon aspect ratio invalid — length (10.0mm) must exceed width (20.0mm)

✅ Input validation confirmed


In [ ]:
import uuid

app_a = FastAPI(
    title       = "Service A — Parameter Generator",
    description = "Generates and validates SiC/SiC composite parameters, "
                  "computes effective modulus via Voigt rule of mixtures, "
                  "and triggers Service B via event-driven POST request.",
    version     = "1.0.0"
)

# ── POST /generate ────────────────────────────────────────────────
@app_a.post('/generate')
def generate_parameters(p: ParameterRequest = None):

    if p is None:
        p = ParameterRequest()

    # Generate unique job ID and record start time
    job_id = str(uuid.uuid4())[:8].upper()
    pipeline_state['job_id']     = job_id
    pipeline_state['started_at'] = datetime.now().isoformat()
    pipeline_state['errors']     = []

    update_state('A_parameter_generation', 'running')
    log_event('Service A', 'INFO',
              f"New job started | job_id={job_id}")

    # Layer 1 — validate inputs
    validate_parameters(p)

    # Compute effective modulus
    E_eff = compute_effective_modulus(
        p.E_fiber_GPa,
        p.E_matrix_GPa,
        p.fiber_volume_fraction
    )

    # Assemble clean payload for Service B
    payload = {
        'job_id'               : job_id,
        'E_GPa'                : E_eff,
        'nu'                   : p.nu,
        'applied_stress'       : p.applied_stress_MPa,
        'length_mm'            : p.length_mm,
        'width_mm'             : p.width_mm,
        'fiber_volume_fraction': p.fiber_volume_fraction,
        'E_fiber_GPa'          : p.E_fiber_GPa,
        'E_matrix_GPa'         : p.E_matrix_GPa,
        'timestamp'            : datetime.now().isoformat()
    }

    log_event('Service A', 'SUCCESS',
              f"Payload assembled | E_eff={E_eff} GPa | job={job_id}")
    update_state('A_parameter_generation', 'complete')

    # ── Event-driven trigger → Service B ─────────────────────────
    try:
        log_event('Service A', 'INFO',
                  f"Triggering Service B | job={job_id}")
        service_b_response = post_with_retry(
            url  = 'http://localhost:8002/simulate',
            data = payload
        )
        return {
            'status'     : 'pipeline_triggered',
            'job_id'     : job_id,
            'parameters' : payload,
            'service_b'  : service_b_response
        }

    except RuntimeError as e:
        # Service B unreachable — log and return payload anyway
        error_msg = str(e)
        update_state('A_parameter_generation', 'error', error_msg)
        log_event('Service A', 'ERROR',
                  f"Service B unreachable | job={job_id} | {error_msg}")
        return {
            'status'    : 'service_b_unreachable',
            'job_id'    : job_id,
            'parameters': payload,
            'error'     : error_msg
        }

# ── GET /health ───────────────────────────────────────────────────
@app_a.get('/health')
def health_a():
    return {
        'service'  : 'A — Parameter Generator',
        'status'   : 'online',
        'port'     : 8001,
        'timestamp': datetime.now().isoformat()
    }

# ── GET /state ────────────────────────────────────────────────────
@app_a.get('/state')
def state_a():
    return {
        'pipeline_state': pipeline_state,
        'timestamp'     : datetime.now().isoformat()
    }

# ── Confirmation ──────────────────────────────────────────────────
print("✅ Service A — FastAPI app defined")
print("\n  Endpoints registered:")
print("    POST /generate → validate, compute E_eff, trigger Service B")
print("    GET  /health   → confirm service is online")
print("    GET  /state    → return current pipeline state")
print("\n  Ready to start on port 8001")

✅ Service A — FastAPI app defined

  Endpoints registered:
    POST /generate → validate, compute E_eff, trigger Service B
    GET  /health   → confirm service is online
    GET  /state    → return current pipeline state

  Ready to start on port 8001


In [ ]:
import asyncio
from uvicorn import Config, Server

async def serve_a():
    config = Config(
        app       = app_a,
        host      = '0.0.0.0',
        port      = 8001,
        log_level = 'warning'
    )
    server = Server(config)
    await server.serve()

def run_service_a():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(serve_a())

# Start in background daemon thread
thread_a = threading.Thread(target=run_service_a, daemon=True)
thread_a.start()

# Wait for service to fully initialise
time.sleep(3)

# Health check
try:
    r = requests.get('http://localhost:8001/health', timeout=5)
    health = r.json()
    print("✅ Service A is online:")
    print(f"   Service  : {health['service']}")
    print(f"   Status   : {health['status']}")
    print(f"   Port     : {health['port']}")
    print(f"   Time     : {health['timestamp']}")
    log_event('Service A', 'INFO', 'Service A started on port 8001')
except Exception as e:
    print(f"❌ Service A health check failed: {e}")

2026-06-01 17:55:46 | INFO | Service A                      | INFO       | Service A started on port 8001
INFO:MCL_Orchestrator:Service A                      | INFO       | Service A started on port 8001


✅ Service A is online:
   Service  : A — Parameter Generator
   Status   : online
   Port     : 8001
   Time     : 2026-06-01T17:55:46.065302


In [ ]:
# Send valid SiC/SiC parameters to Service A
test_params = {
    'E_fiber_GPa'          : 380.0,
    'E_matrix_GPa'         : 90.0,
    'fiber_volume_fraction': 0.45,
    'nu'                   : 0.20,
    'applied_stress_MPa'   : 300.0,
    'length_mm'            : 100.0,
    'width_mm'             : 20.0
}

print("Test 1 — Valid SiC/SiC Parameters")
print("=" * 50)

response = requests.post(
    'http://localhost:8001/generate',
    json    = test_params,
    timeout = 60
)

result = response.json()

print(f"  HTTP Status   : {response.status_code}")
print(f"  Pipeline status: {result['status']}")
print(f"  Job ID        : {result['job_id']}")
print(f"\n  Generated Payload:")
print(f"  " + "-" * 45)
for key, value in result['parameters'].items():
    print(f"    {key:<25} : {value}")
print(f"  " + "-" * 45)

# Confirm Service B trigger was attempted
if result['status'] == 'service_b_unreachable':
    print(f"\n  ⚠️  Service B not running yet — expected at this stage")
    print(f"  ✅ Service A handled the missing Service B gracefully")
    print(f"  Error logged : {result['error']}")
elif result['status'] == 'pipeline_triggered':
    print(f"\n  ✅ Full pipeline triggered successfully")

print(f"\n✅ Test 1 complete — Service A working correctly")

2026-06-01 17:57:20 | INFO | A_parameter_generation         | running    | Stage updated to running
INFO:MCL_Orchestrator:A_parameter_generation         | running    | Stage updated to running


Test 1 — Valid SiC/SiC Parameters


2026-06-01 17:57:20 | INFO | Service A                      | INFO       | New job started | job_id=9EB0728C
INFO:MCL_Orchestrator:Service A                      | INFO       | New job started | job_id=9EB0728C
2026-06-01 17:57:20 | INFO | Service A — validation         | SUCCESS    | All parameters within physical range
INFO:MCL_Orchestrator:Service A — validation         | SUCCESS    | All parameters within physical range
2026-06-01 17:57:21 | INFO | Service A                      | SUCCESS    | Payload assembled | E_eff=220.5 GPa | job=9EB0728C
INFO:MCL_Orchestrator:Service A                      | SUCCESS    | Payload assembled | E_eff=220.5 GPa | job=9EB0728C
2026-06-01 17:57:21 | INFO | A_parameter_generation         | complete   | Stage updated to complete
INFO:MCL_Orchestrator:A_parameter_generation         | complete   | Stage updated to complete
2026-06-01 17:57:21 | INFO | Service A                      | INFO       | Triggering Service B | job=9EB0728C
INFO:MCL_Orchestrator

  ⚠️  Service unreachable — retrying in 1s (attempt 1/3)


2026-06-01 17:57:22 | WARNING | retry_handler                  | WARNING    | Connection failed — retrying in 2s


  ⚠️  Service unreachable — retrying in 2s (attempt 2/3)


2026-06-01 17:57:24 | ERROR | retry_handler                  | ERROR      | Unreachable after 3 attempts
ERROR:MCL_Orchestrator:retry_handler                  | ERROR      | Unreachable after 3 attempts
2026-06-01 17:57:24 | ERROR | A_parameter_generation         | ERROR      | ❌ Service at http://localhost:8002/simulate unreachable after 3 attempts
ERROR:MCL_Orchestrator:A_parameter_generation         | ERROR      | ❌ Service at http://localhost:8002/simulate unreachable after 3 attempts
2026-06-01 17:57:24 | ERROR | Service A                      | ERROR      | Service B unreachable | job=9EB0728C | ❌ Service at http://localhost:8002/simulate unreachable after 3 attempts
ERROR:MCL_Orchestrator:Service A                      | ERROR      | Service B unreachable | job=9EB0728C | ❌ Service at http://localhost:8002/simulate unreachable after 3 attempts


  HTTP Status   : 200
  Pipeline status: service_b_unreachable
  Job ID        : 9EB0728C

  Generated Payload:
  ---------------------------------------------
    job_id                    : 9EB0728C
    E_GPa                     : 220.5
    nu                        : 0.2
    applied_stress            : 300.0
    length_mm                 : 100.0
    width_mm                  : 20.0
    fiber_volume_fraction     : 0.45
    E_fiber_GPa               : 380.0
    E_matrix_GPa              : 90.0
    timestamp                 : 2026-06-01T17:57:21.004517
  ---------------------------------------------

  ⚠️  Service B not running yet — expected at this stage
  ✅ Service A handled the missing Service B gracefully
  Error logged : ❌ Service at http://localhost:8002/simulate unreachable after 3 attempts

✅ Test 1 complete — Service A working correctly


In [ ]:
# Send deliberately invalid parameters
invalid_params = {
    'E_fiber_GPa'          : 999.0,    # too high
    'E_matrix_GPa'         : 90.0,
    'fiber_volume_fraction': 0.95,     # out of range
    'nu'                   : 0.20,
    'applied_stress_MPa'   : -100.0,   # negative
    'length_mm'            : 10.0,     # shorter than width
    'width_mm'             : 20.0
}

print("Test 2 — Invalid Parameters")
print("=" * 50)

response = requests.post(
    'http://localhost:8001/generate',
    json    = invalid_params,
    timeout = 30
)

print(f"  HTTP Status : {response.status_code}")

if response.status_code == 422:
    print(f"  ✅ Correctly rejected with 422 Unprocessable Entity")
    print(f"\n  Validation errors returned:")
    errors = response.json().get('detail', [])
    for i, error in enumerate(errors, 1):
        print(f"    {i}. {error}")
else:
    print(f"  ❌ Unexpected status — expected 422, got {response.status_code}")

# Confirm service is still running after bad request
print(f"\n  Confirming Service A still online after bad request:")
health = requests.get('http://localhost:8001/health', timeout=5).json()
print(f"  Status : {health['status']} ✅")

print(f"\n✅ Test 2 complete — error handling confirmed")

2026-06-01 17:59:15 | INFO | A_parameter_generation         | running    | Stage updated to running
INFO:MCL_Orchestrator:A_parameter_generation         | running    | Stage updated to running
2026-06-01 17:59:15 | INFO | Service A                      | INFO       | New job started | job_id=84BE57B5
INFO:MCL_Orchestrator:Service A                      | INFO       | New job started | job_id=84BE57B5
2026-06-01 17:59:15 | ERROR | Service A — validation         | ERROR      | 4 validation error(s) detected
ERROR:MCL_Orchestrator:Service A — validation         | ERROR      | 4 validation error(s) detected


Test 2 — Invalid Parameters
  HTTP Status : 422
  ✅ Correctly rejected with 422 Unprocessable Entity

  Validation errors returned:
    1. E_fiber_GPa=999.0 out of range — expected 100-500 GPa for ceramic fibers
    2. fiber_volume_fraction=0.95 out of range — expected 0.3-0.7 for SiC/SiC composites
    3. applied_stress_MPa=-100.0 invalid — must be positive for a tensile test
    4. Coupon aspect ratio invalid — length (10.0mm) must exceed width (20.0mm)

  Confirming Service A still online after bad request:
  Status : online ✅

✅ Test 2 complete — error handling confirmed


In [ ]:
# Check pipeline state
state_response = requests.get(
    'http://localhost:8001/state',
    timeout=5
).json()

current_state = state_response['pipeline_state']

print("=" * 55)
print("  SERVICE A — COMPLETE")
print("=" * 55)
print(f"  Timestamp     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Port          : 8001")
print(f"  Last Job ID   : {current_state['job_id']}")
print(f"  Started at    : {current_state['started_at']}")

print(f"\n  Stage Status:")
for stage, status in current_state['stages'].items():
    icon = "✅" if status == 'complete' else \
           "❌" if status in ['error', 'failed'] else "⏳"
    print(f"    {icon} {stage:<35} : {status}")

print(f"\n  Endpoints confirmed:")
print(f"    ✅ POST /generate — validated and working")
print(f"    ✅ GET  /health   — responding correctly")
print(f"    ✅ GET  /state    — returning pipeline state")

print(f"\n  Payload sent to Service B:")
print(f"  " + "-" * 45)
payload_fields = {
    'job_id'               : 'Unique pipeline run identifier',
    'E_GPa'                : 'Effective modulus (Voigt)',
    'nu'                   : "Poisson's ratio",
    'applied_stress'       : 'Applied tensile stress (MPa)',
    'length_mm'            : 'Coupon length',
    'width_mm'             : 'Coupon width',
    'fiber_volume_fraction': 'Fiber volume fraction',
    'E_fiber_GPa'          : 'Raw fiber modulus (for traceability)',
    'E_matrix_GPa'         : 'Raw matrix modulus (for traceability)',
    'timestamp'            : 'Job creation time'
}
for field, description in payload_fields.items():
    print(f"    {field:<25} : {description}")
print(f"  " + "-" * 45)

print(f"\n  Errors logged : {len(current_state['errors'])}")
print("=" * 55)
print("  ✅ Ready to proceed to Notebook 3 — Service B")
print("=" * 55)

log_event('01_service_A', 'INFO',
          'Service A notebook complete — all tests passed')

2026-06-01 17:59:54 | INFO | 01_service_A                   | INFO       | Service A notebook complete — all tests passed
INFO:MCL_Orchestrator:01_service_A                   | INFO       | Service A notebook complete — all tests passed


  SERVICE A — COMPLETE
  Timestamp     : 2026-06-01 17:59:54
  Port          : 8001
  Last Job ID   : 84BE57B5
  Started at    : 2026-06-01T17:59:15.677981

  Stage Status:
    ⏳ A_parameter_generation              : running
    ⏳ B_fem_simulation                    : pending
    ⏳ C_postprocessing                    : pending

  Endpoints confirmed:
    ✅ POST /generate — validated and working
    ✅ GET  /health   — responding correctly
    ✅ GET  /state    — returning pipeline state

  Payload sent to Service B:
  ---------------------------------------------
    job_id                    : Unique pipeline run identifier
    E_GPa                     : Effective modulus (Voigt)
    nu                        : Poisson's ratio
    applied_stress            : Applied tensile stress (MPa)
    length_mm                 : Coupon length
    width_mm                  : Coupon width
    fiber_volume_fraction     : Fiber volume fraction
    E_fiber_GPa               : Raw fiber modulus (for tr